In [1]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)

In [2]:
# Load draft data and player data

df_draft_data = pd.read_csv('2024_draft_leagues_1283_recap_combined.csv')
df_coach_list = pd.read_csv('../../inputs/coach_list.csv', index_col=0)
df_player_data = pd.read_csv('2024_player_stats_current.csv')

In [3]:
# Calculate player stats
rnds = df_player_data['played'].max()
df_player_data['avg_adj'] = round(((df_player_data['avg'] * df_player_data['played']) + ((rnds - df_player_data['played']) * np.where(df_player_data['avg'] < 75, df_player_data['avg'], 75))) / rnds, 1)
# df_player_data[df_player_data['last_name'] == "Bontempelli"]
df_player_data

,feed_id,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round,points,played,avg,avg3,avg5,price,avg_adj
0,1012807,1,Sam,Berry,ADE,MID,NaN,1,80,1,80.0000,80.0000,80.00,226900,75.2
1,1012807,1,Sam,Berry,ADE,MID,NaN,2,52,2,66.0000,66.0000,66.00,226900,66.0
2,1012807,1,Sam,Berry,ADE,MID,NaN,4,33,3,55.0000,55.0000,55.00,243100,55.0
3,1012807,1,Sam,Berry,ADE,MID,NaN,5,56,4,55.2500,47.0000,55.25,244400,55.2
4,1012807,1,Sam,Berry,ADE,MID,NaN,6,31,5,50.4000,40.0000,50.40,235500,50.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9512,996731,99,Charlie,Curnow,CAR,FWD,NaN,18,89,17,85.6471,58.6667,74.80,394800,82.9
9513,996731,99,Charlie,Curnow,CAR,FWD,NaN,19,106,18,86.7778,76.0000,80.00,393700,84.2
9514,996731,99,Charlie,Curnow,CAR,FWD,NaN,20,119,19,88.4737,104.6670,80.20,431500,86.1
9515,996731,99,Charlie,Curnow,CAR,FWD,NaN,21,33,20,85.7000,86.0000,76.00,433900,84.3


In [4]:
# Standardize averages
pos_list = pd.DataFrame({
    'p': ['DEF', 'MID', 'RUC', 'FWD'],
    'n': [5, 7, 2, 5]
})
pos_list['sum_n'] = pos_list['n'].sum()
pos_list['weight'] = round(pos_list['n'] / pos_list['sum_n'], 3)
pos_list


,p,n,sum_n,weight
0,DEF,5,19,0.263
1,MID,7,19,0.368
2,RUC,2,19,0.105
3,FWD,5,19,0.263


In [5]:
scr_data = pd.DataFrame()
overall_smy = pd.DataFrame()

for i in range(len(pos_list)):
    p = pos_list.loc[i, 'p']
    n = pos_list.loc[i, 'n']
    
    pos_data = df_player_data[(df_player_data['pos_1'].str.contains(p)) | (df_player_data['pos_2'].str.contains(p))].copy()
    
    pos_smy = pos_data[['feed_id', 'avg_adj']].sort_values(by='avg_adj', ascending=False).head(n * 8)
    
    mean = pos_smy['avg_adj'].mean()
    sd = pos_smy['avg_adj'].std()
    
    pos_data.loc[:, 'pos_scr'] = round((pos_data['avg_adj'] - mean) / sd, 3)
    pos_data.loc[:, 'scr_pos'] = p
    
    scr_data = pd.concat([scr_data, pos_data[['feed_id', 'pos_scr', 'scr_pos']]])
    overall_smy = pd.concat([overall_smy, pos_smy])

scr_data = scr_data.sort_values(by='pos_scr', ascending=False).drop_duplicates(subset='feed_id')
scr_data

,feed_id,pos_scr,scr_pos
8034,297373,3.304,MID
524,1023261,2.352,DEF
2403,1009260,2.201,FWD
1965,998659,2.196,DEF
5822,1006121,1.821,MID
...,...,...,...
44,295518,-21.758,MID
660,1032017,-21.974,DEF
6990,1027872,-22.366,DEF
2546,996464,-24.323,DEF


In [6]:
mean = overall_smy['avg_adj'].mean()
display("mean: " + str(mean))
sd = overall_smy['avg_adj'].std()
display("Standard Deviation: " + str(sd))

'mean: 109.29342105263157'

'Standard Deviation: 5.973137117699427'

In [7]:
# add mean and standard deviation to df_player_data
df_player_data['scr'] = round((df_player_data['avg_adj'] - mean) / sd, 3)
df_player_data

,feed_id,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round,points,played,avg,avg3,avg5,price,avg_adj,scr
0,1012807,1,Sam,Berry,ADE,MID,NaN,1,80,1,80.0000,80.0000,80.00,226900,75.2,-5.708
1,1012807,1,Sam,Berry,ADE,MID,NaN,2,52,2,66.0000,66.0000,66.00,226900,66.0,-7.248
2,1012807,1,Sam,Berry,ADE,MID,NaN,4,33,3,55.0000,55.0000,55.00,243100,55.0,-9.090
3,1012807,1,Sam,Berry,ADE,MID,NaN,5,56,4,55.2500,47.0000,55.25,244400,55.2,-9.056
4,1012807,1,Sam,Berry,ADE,MID,NaN,6,31,5,50.4000,40.0000,50.40,235500,50.4,-9.860
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9512,996731,99,Charlie,Curnow,CAR,FWD,NaN,18,89,17,85.6471,58.6667,74.80,394800,82.9,-4.419
9513,996731,99,Charlie,Curnow,CAR,FWD,NaN,19,106,18,86.7778,76.0000,80.00,393700,84.2,-4.201
9514,996731,99,Charlie,Curnow,CAR,FWD,NaN,20,119,19,88.4737,104.6670,80.20,431500,86.1,-3.883
9515,996731,99,Charlie,Curnow,CAR,FWD,NaN,21,33,20,85.7000,86.0000,76.00,433900,84.3,-4.184


In [8]:
df_player_data = df_player_data.merge(scr_data, on='feed_id', how='left')
df_player_data

,feed_id,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round,points,played,avg,avg3,avg5,price,avg_adj,scr,pos_scr,scr_pos
0,1012807,1,Sam,Berry,ADE,MID,NaN,1,80,1,80.0000,80.0000,80.00,226900,75.2,-5.708,-9.351,MID
1,1012807,1,Sam,Berry,ADE,MID,NaN,2,52,2,66.0000,66.0000,66.00,226900,66.0,-7.248,-9.351,MID
2,1012807,1,Sam,Berry,ADE,MID,NaN,4,33,3,55.0000,55.0000,55.00,243100,55.0,-9.090,-9.351,MID
3,1012807,1,Sam,Berry,ADE,MID,NaN,5,56,4,55.2500,47.0000,55.25,244400,55.2,-9.056,-9.351,MID
4,1012807,1,Sam,Berry,ADE,MID,NaN,6,31,5,50.4000,40.0000,50.40,235500,50.4,-9.860,-9.351,MID
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9512,996731,99,Charlie,Curnow,CAR,FWD,NaN,18,89,17,85.6471,58.6667,74.80,394800,82.9,-4.419,-2.952,FWD
9513,996731,99,Charlie,Curnow,CAR,FWD,NaN,19,106,18,86.7778,76.0000,80.00,393700,84.2,-4.201,-2.952,FWD
9514,996731,99,Charlie,Curnow,CAR,FWD,NaN,20,119,19,88.4737,104.6670,80.20,431500,86.1,-3.883,-2.952,FWD
9515,996731,99,Charlie,Curnow,CAR,FWD,NaN,21,33,20,85.7000,86.0000,76.00,433900,84.3,-4.184,-2.952,FWD


In [9]:
df_player_data = df_player_data.merge(pos_list[['p', 'weight']], left_on='scr_pos', right_on='p', how='left')
df_player_data

,feed_id,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round,points,played,avg,avg3,avg5,price,avg_adj,scr,pos_scr,scr_pos,p,weight
0,1012807,1,Sam,Berry,ADE,MID,NaN,1,80,1,80.0000,80.0000,80.00,226900,75.2,-5.708,-9.351,MID,MID,0.368
1,1012807,1,Sam,Berry,ADE,MID,NaN,2,52,2,66.0000,66.0000,66.00,226900,66.0,-7.248,-9.351,MID,MID,0.368
2,1012807,1,Sam,Berry,ADE,MID,NaN,4,33,3,55.0000,55.0000,55.00,243100,55.0,-9.090,-9.351,MID,MID,0.368
3,1012807,1,Sam,Berry,ADE,MID,NaN,5,56,4,55.2500,47.0000,55.25,244400,55.2,-9.056,-9.351,MID,MID,0.368
4,1012807,1,Sam,Berry,ADE,MID,NaN,6,31,5,50.4000,40.0000,50.40,235500,50.4,-9.860,-9.351,MID,MID,0.368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9512,996731,99,Charlie,Curnow,CAR,FWD,NaN,18,89,17,85.6471,58.6667,74.80,394800,82.9,-4.419,-2.952,FWD,FWD,0.263
9513,996731,99,Charlie,Curnow,CAR,FWD,NaN,19,106,18,86.7778,76.0000,80.00,393700,84.2,-4.201,-2.952,FWD,FWD,0.263
9514,996731,99,Charlie,Curnow,CAR,FWD,NaN,20,119,19,88.4737,104.6670,80.20,431500,86.1,-3.883,-2.952,FWD,FWD,0.263
9515,996731,99,Charlie,Curnow,CAR,FWD,NaN,21,33,20,85.7000,86.0000,76.00,433900,84.3,-4.184,-2.952,FWD,FWD,0.263


In [10]:
df_player_data['scr_wgt'] = df_player_data['pos_scr'] * df_player_data['weight'] + df_player_data['scr'] * (1 - df_player_data['weight'])
df_player_data = df_player_data.sort_values(by='scr_wgt', ascending=False)
df_player_data['rank'] = df_player_data.index + 1
df_player_data['name'] = df_player_data.apply(lambda row: f"{row.first_name[0]}.{row.last_name}", axis=1)
df_player_data

,feed_id,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round,points,played,avg,avg3,avg5,price,avg_adj,scr,pos_scr,scr_pos,p,weight,scr_wgt,rank,name
8034,297373,695,Marcus,Bontempelli,WBD,MID,NaN,24,116,23,126.391,119.333,129.8,665800,126.4,2.864,3.304,MID,MID,0.368,3.025920,8035,M.Bontempelli
8033,297373,695,Marcus,Bontempelli,WBD,MID,NaN,23,123,22,126.864,141.333,131.0,685300,124.6,2.563,3.304,MID,MID,0.368,2.835688,8034,M.Bontempelli
8032,297373,695,Marcus,Bontempelli,WBD,MID,NaN,22,119,21,127.048,136.667,128.0,670200,122.5,2.211,3.304,MID,MID,0.368,2.613224,8033,M.Bontempelli
8031,297373,695,Marcus,Bontempelli,WBD,MID,NaN,21,182,20,127.450,137.667,125.4,657700,120.6,1.893,3.304,MID,MID,0.368,2.412248,8032,M.Bontempelli
8030,297373,695,Marcus,Bontempelli,WBD,MID,NaN,20,109,19,124.579,113.000,120.4,640500,116.0,1.123,3.304,MID,MID,0.368,1.925608,8031,M.Bontempelli
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1500,1012817,201,Zach,Reid,ESS,DEF,NaN,1,15,1,15.000,15.000,15.0,123900,15.0,-15.786,-24.323,DEF,DEF,0.263,-18.031231,1501,Z.Reid
6747,1022999,599,Kaleb,Smith,RIC,DEF,NaN,12,5,1,5.000,5.000,5.0,123900,5.0,-17.460,-20.147,DEF,DEF,0.263,-18.166681,6748,K.Smith
6988,1027872,618,Angus,Hastie,STK,DEF,NaN,4,1,2,9.500,9.500,9.5,117300,9.5,-16.707,-22.366,DEF,DEF,0.263,-18.195317,6989,A.Hastie
3116,1021103,328,Mitchell,Knevitt,GEE,MID,NaN,2,0,1,0.000,0.000,0.0,306000,0.0,-18.297,-18.051,MID,MID,0.368,-18.206472,3117,M.Knevitt


In [12]:
df_draft_data = df_draft_data.merge(df_player_data[['player_id', 'avg_adj', 'scr_wgt', 'rank']], on='player_id', how='left').sort_values(by=['round', 'pick', 'scr_wgt'], ascending=[True, True, False])
df_draft_data


,id,league_id,user_team_id,round,pick,player_id,position,autopicked,time,avg_adj,scr_wgt,rank
0,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,126.4,3.025920,8035
1,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,124.6,2.835688,8034
2,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,122.5,2.613224,8033
3,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,120.6,2.412248,8032
4,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,116.0,1.925608,8031
...,...,...,...,...,...,...,...,...,...,...,...,...
3532,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.4,-7.636588,920
3533,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.3,-7.649117,926
3534,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.2,-7.660909,921
3535,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,58.5,-7.747875,915


In [13]:
df_draft_data['roll_rank'] = df_draft_data.groupby('round')['scr_wgt'].rank(method='first')
df_draft_data

,id,league_id,user_team_id,round,pick,player_id,position,autopicked,time,avg_adj,scr_wgt,rank,roll_rank
0,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,126.4,3.025920,8035,169.0
1,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,124.6,2.835688,8034,168.0
2,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,122.5,2.613224,8033,167.0
3,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,120.6,2.412248,8032,166.0
4,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,116.0,1.925608,8031,165.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3532,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.4,-7.636588,920,37.0
3533,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.3,-7.649117,926,36.0
3534,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.2,-7.660909,921,35.0
3535,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,58.5,-7.747875,915,34.0


In [15]:

df_draft_data = df_draft_data.merge(df_coach_list, left_on='user_team_id', right_on='coach_team_id', how='left')
df_draft_data


,id,league_id,user_team_id,round,pick,player_id,position,autopicked,time,avg_adj,scr_wgt,rank,roll_rank,coach_id,coach_team_id,coach_first_name,coach_team_name
0,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,126.4,3.025920,8035,169.0,21562,714,Mark,Need for Sheed
1,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,124.6,2.835688,8034,168.0,21562,714,Mark,Need for Sheed
2,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,122.5,2.613224,8033,167.0,21562,714,Mark,Need for Sheed
3,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,120.6,2.412248,8032,166.0,21562,714,Mark,Need for Sheed
4,1705893,1283,714,1,1,695,MID,0,8/03/2024 13:18,116.0,1.925608,8031,165.0,21562,714,Mark,Need for Sheed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3532,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.4,-7.636588,920,37.0,180650,720,Jordan,StupidSexyFlanders
3533,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.3,-7.649117,926,36.0,180650,720,Jordan,StupidSexyFlanders
3534,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,59.2,-7.660909,921,35.0,180650,720,Jordan,StupidSexyFlanders
3535,1736462,1283,720,23,184,167,FWD,0,9/03/2024 13:08,58.5,-7.747875,915,34.0,180650,720,Jordan,StupidSexyFlanders


In [16]:
# Calculate Draft data
max_round = int(df_draft_data['round'].max())
asl_order = list(range(1, 9)) + (list(range(1, 9)) + list(range(1, 9))[::-1]) * (int(df_draft_data['round'].max()))
df_draft_data['asl_order'] = asl_order[:len(df_draft_data)]
df_draft_data = df_draft_data.sort_values(by=['round', 'asl_order'])
df_draft_data['asl_pick'] = df_draft_data.index + 1

# Display the first few rows of the sorted draft data
df_draft_data

ValueError: Length of values (376) does not match length of index (3537)

In [ ]:
draft_raw.pivot(index='round', columns='coach_first_name')

In [ ]:
draft_table_color = draft_raw.pivot(index='round', columns='coach_first_name', values='roll_rank').reset_index()
draft_table_color.columns.name = None

In [ ]:
draft_table = draft_raw.pivot(index='round', columns='coach', values='name').reset_index()
draft_table.columns.name = None
draft_table_color.columns.name = None

In [ ]:
# Display the tables
print(draft_table)
print(draft_table_color)

In [ ]:
# -- Save Results
# Save the final dataset to a CSV
# heat_map_data.to_csv('2024_draft_heat_map.csv', index=False)